In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
import torch
from darts.utils.missing_values import extract_subseries
import darts
from aare.constants import TEMP

import pandas as pd
from aare_train.fetching.AareDataset import AareDataset

from aare_train.wrappers.arima_fixed import ARIMAFix
from aare_train.evaluation.evaluation import evaluate_model
from aare_train.params import read_params
from aare_train.preparation import (
    prepare_ts_aare_temp,
    resample,
    interpolate_aare_temp,
)
from aare_influx.remote_existenz_store import RemoteExistenzStore
from aare_train.darts_utils import to_ts, get_context_len

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
store = RemoteExistenzStore()
ds = AareDataset.from_conf()

In [ ]:
train = prepare_ts_aare_temp(ds.get_train())
train

# ARIMA(X) (with just air temp)

Reasons I think this might work:

1. The water temperatures seems to follow an AR(1) process and is stationary after first difference
2. The air temperature (forecast), which is probably our most dominant covariate/predictor, captures the same seasonality as the water temperature (daily and yearly), so we don't need to worry about seasonality. This is an especially important point because ARIMA cannot handle multiple seasonalities (should use TBATS instead for example).

Reasons this might not perform very well:

1. The relationship between water and air temperature is non-linear at low and high temperatures (DOI 10.1029/98WR01877). Could use some non-linear transformations of the air temp as covariates to help with this.
2. We already know lag 1 (hour) of the air temperature is the best lag (highest correlation), so there is a seasonality discrepancy of 1 hour
3. There are other factors that influence the water temperature and the relationships and interactions are probably much more complex than ARIMAX can model.


In [ ]:
exp = {}

In [ ]:
# just as a first
arima = ARIMAFix(p=1, d=1, q=0)  # use our own class for better extreme_lags
arima

In [ ]:
arima.fit(train)  # this can and will contain gaps, but ARIMA seemingly has no issues with that?

In [ ]:
validation_params = params["validation"]
stride = validation_params["stride"]
min_lookback_hours = validation_params["min_lookback_hours"]
forecast_horizon = params["general"]["forecast_horizon"]

In [ ]:
val = prepare_ts_aare_temp(ds.get_val())
val

In [ ]:
val_subs = extract_subseries(val, mode="any")
val_l = sorted(val_subs, key=len, reverse=True)[0]
val_l

In [ ]:
# just validate on the longest chunk. already takes 11min
# metrics, sample = evaluate_model(arima, val_l, forecast_horizon, stride, parallel=False, min_lookback_hours=24, verbose=True)
# print(metrics)
# _ = sample.plot("ARIMA [longest]")

In [ ]:
short_subs = [v.drop_after(200 * (i + 1)) for i, v in enumerate(val_subs)]
short_subs

In [ ]:
# THIS TOOK 24m to run!!! TODOne implement parallelized evaluation and do it again :) also importing a different func now cuz rename
# ALSO TODOne: What actually happens when you evaluate a local forecasting model like this? The historical forecast method says that
# retrain=False is only supported for global models, but is it?
# LMAO the docs are wrong. There are TransferableFutureCovariatesLocalForecastingModel (like our ARIMA),
# which interally set _supports_non_retrainable_historical_forecasts to true, so actually some local models
# work too, hooray. Could even submit an issue. Ps. the public property seems to be supports_transferrable_series_prediction,
# but they are only similar semantically and not bound together programmatically (not sure why).

# Indeed, implementing parallelization cuts the time down to the 11min needed for the longest slice.
# Of course, slicing at places without gaps to get smaller chunks would give even better performance.
metrics, sample = evaluate_model(
    arima,
    val_subs,  # short_subs,  <-- for testing and debugging the parallelization
    forecast_horizon,
    stride,
    parallel=True,
    min_lookback_hours=24,
    verbose=True,
)

In [ ]:
exp["simple"] = {
    "model": arima,
    "metrics": metrics,
    "sample": sample,
}

In [ ]:
sample.plot("ARIMA")
metrics

In [ ]:
get_context_len(arima)

In [ ]:
# Oh and another TODOne!
# Analyze how to get the correct context length of a model, maybe you need special cases for some classes like ARIMA. we care about the predict-time context-length.
# https://github.com/unit8co/darts/blob/42776790183bbe42411fc2dba3e1e8416f9263e8/darts/models/forecasting/forecasting_model.py#L3360
# ARIMA just hard-codes 30 as their min train length, which cannot be the actual context len: https://github.com/unit8co/darts/blob/42776790183bbe42411fc2dba3e1e8416f9263e8/darts/models/forecasting/arima.py#L233
# It seems that the models that we can work with are "transferrable" models, and all global models are tranferrable, but only few
# local ones are.
# TODOne figure out if there's a common base class we could use, but probably not -> yeah no
# TODOne also add a verbose flag to the evaluation
# ALSO: ARIMA supports probabilistic forecasts and we love that, so make sure to also add that to the eval pipeline sometime
# and think about where to put the number of samples [params] (i.e. is that a general setting for all models, one per model, etc).

# Searching the Darts codebase, the following models support transferrable series prediction:
# - GlobalForecastingModel and all derivatives
# - TransferableFutureCovariateLocalForecastingModel and all derivatives, which currently are ARIMA, VARIMA and KalmanForecaster

# Similarly, the following models support non retrainable historical forecasts
# - GlobalForecastingModel and all derivatives
# - TransferableFutureCovariateLocalForecastingModel and all derivatives
# - Global ensemble models

# So only ensembles are different, but interestingly, EnsembleModel is a global model, so a global model always supports transferrable series prediction,
# but it only supports non-retrainable historical forecasts if all models in the ensemble are global models.

# ARIMA with time as cov

In [ ]:
arima = ARIMAFix(p=1, d=1, q=0, add_encoders={"cyclic": {"future": ["hour"]}})
arima.fit(train)

In [ ]:
metrics, sample = evaluate_model(
    arima,
    val_subs,
    forecast_horizon,
    stride,
    parallel=True,
    min_lookback_hours=24,
    verbose=True,
)

In [ ]:
exp["time"] = {
    "model": arima,
    "metrics": metrics,
    "sample": sample,
}

In [ ]:
sample.plot("ARIMA w/ time")
metrics

# ARIMA with air temperature as cov

In [ ]:
tt_bern = store.query((params["split"]["train_split"], params["split"]["test_split"]), "smn/tt:mean_1h@bern")
tt_bern

In [ ]:
tt_bern = resample(tt_bern)
# TODO remove_outliers
tt_bern = interpolate_aare_temp(tt_bern, drop_filled=True, columns="tt_bern")
tt_bern = to_ts(tt_bern, col="tt_bern")
tt_bern

In [ ]:
tt_bern_train, tt_bern_val = tt_bern.split_after(pd.Timestamp(params["split"]["val_split"]))
(len(tt_bern_train), len(tt_bern_val))

In [ ]:
# we lose a lot of data because tt is only available from 2013
train_combined = darts.concatenate([train.slice_intersect(tt_bern_train), tt_bern_train], axis="component")

In [ ]:
train_combined

In [ ]:
train_combined_subs = extract_subseries(train_combined, mode="any")

In [ ]:
len(train_combined_subs)

In [ ]:
train_combined_l = sorted(train_combined_subs, key=len, reverse=True)[0]

In [ ]:
assert train_combined_l.gaps().empty, "train_combined_l has gaps?!"

In [ ]:
train_combined_l

In [ ]:
arima = ARIMAFix(p=1, d=1, q=0)  # , add_encoders={'cyclic': {'future': ['hour']}})
arima.fit(train_combined_l[TEMP], future_covariates=train_combined_l["tt_bern"])

In [ ]:
val_combined = darts.concatenate([val, tt_bern_val], axis="component")
val_combined_subs = extract_subseries(val_combined, mode="any")

In [ ]:
metrics, sample = evaluate_model(
    arima,
    [ts[TEMP] for ts in val_combined_subs],
    forecast_horizon,
    stride,
    parallel=True,
    min_lookback_hours=24,
    verbose=True,
    future_cov=tt_bern_val,
)

In [ ]:
exp["tt_fc"] = {
    "model": arima,
    "metrics": metrics,
    "sample": sample,
}

In [ ]:
sample.plot("ARIMA w/ tt_bern")
metrics

In [ ]:
arima = ARIMAFix(p=1, d=1, q=0, add_encoders={"cyclic": {"future": ["hour"]}})
arima.fit(train_combined_l[TEMP], future_covariates=train_combined_l["tt_bern"])

In [ ]:
metrics, sample = evaluate_model(
    arima,
    [ts[TEMP] for ts in val_combined_subs],
    forecast_horizon,
    stride,
    parallel=True,
    min_lookback_hours=24,
    verbose=True,
    future_cov=tt_bern_val,
)

In [ ]:
exp["time-tt_fc"] = {
    "model": arima,
    "metrics": metrics,
    "sample": sample,
}

In [ ]:
sample.plot("ARIMA w/ time & tt_bern")
metrics

In [ ]:
# store results to disk, even though they're not great.
# not storing the trained model atm, because that's quick to train.

import pickle
from aare_train.paths import METRICS_FOLDER, FORECAST_SAMPLES_FOLDER

import json

for key, val in exp.items():
    name = f"ARIMA_{key}"
    with open(METRICS_FOLDER / f"{name}.json", "wt") as metrics_file:
        json.dump(val["metrics"].to_dict(), metrics_file)

    with open(FORECAST_SAMPLES_FOLDER / f"{name}.pkl", "wb") as forecast_sample_file:
        pickle.dump(val["sample"], forecast_sample_file)

# Conclusion

I will not pursue ARIMA because its performance in these simple tests was worse than both the SNAIVE and TimesFM baselines.

Still worth noting, I just wanted to rush to get a result here and didn't properly change the train/val split as we're now losing ~10 years of data if we include the air temperature.
Also, I didn't ensure reproducibility by setting random states, etc. If this initial testing showed promising results, I would have extracted it into a clean pipeline for extensive training and tuning.
As expected though, these results don't justify spending more time on this ARIMA implementation, and I'd rather try other approaches now.
